# Overview

This notebook is designed for the CatBoost model training.
We will use gradient boosting with extracted images embeddings in order to predict the litotypes on the source images.

Unfortunately, CatBoost doesn't natively support MPS, so CPU calculations will be used instead. 

## 1. Imports and Settings

In [19]:
import ast
import datetime
import os
import warnings

import pandas as pd
import numpy as np
from catboost import CatBoostRegressor
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

In [2]:
BASE_FOLDER = "/Users/locked15/Repos.localized/Python/Study/UUST/Course 3/Semester 2/Practice/sludge-utilities/apps/gb/gb-training_lab/"

DATASET_FOLDER = "data/interim/"
DATASET_TARGET_FILE = "metadata_dinov3_embeddings.parquet"

FEATURE_COLS = [
    "interval_start", "interval_end"
]
TARGET_COLS = [
    'sandstone_sludge', 'siltstone_sludge', 'argillite_sludge'
]

OUTPUT_FOLDER = "output/models/"
MODEL_NAME = "catboost_model-{0}.cbm"

In [3]:
warnings.filterwarnings("ignore")

np.random.seed(42)

## 2. Dataset Loading

In [ ]:
df = pd.read_parquet(os.path.join(BASE_FOLDER, DATASET_FOLDER, DATASET_TARGET_FILE))


print("Dataset loaded. Sample: ")
print(df.head())

Dataset loaded. Sample: 
   well_id  device_no  interval_start  interval_end  sandstone_sludge  \
0        4        1.0            2675          2680                 5   
1        4        2.0            2680          2685                 5   
2        4        3.0            2685          2690                 5   
3        4        4.0            2690          2695                 5   
4        4        5.0            2695          2700                 5   

   siltstone_sludge  argillite_sludge  radiolarite_sludge  coal_sludge  \
0                25                70                   0            0   
1                25                70                   0            0   
2                30                65                   0            0   
3                30                65                   0            0   
4                25                70                   0            0   

   limestone_sludge  ...  oil_saturation  calcite_carbonatometry  \
0                 0  ..

## 3. Embeddings Processing

In [5]:
def str_to_array(x):
    if isinstance(x, str):
        try:
            return np.array(ast.literal_eval(x))
        except:
            return np.array(x)
    return x

df['lba_dinov3_emb'] = df['lba_dinov3_emb'].apply(str_to_array)
df['sludge_dinov3_emb'] = df['sludge_dinov3_emb'].apply(str_to_array)

print("Embeddings formatted as arrays.")
print(df['sludge_dinov3_emb'].iloc[0].shape)

Embeddings formatted as arrays.
(1024,)


## 4. Features and Targets Preparation

In [6]:
lba_emb_df = pd.DataFrame(
    df['lba_dinov3_emb'].tolist(),
    columns = [f"lba_emb_{i}" \
               for i in range(df["lba_dinov3_emb"].iloc[0] \
                                                  .shape[0])
              ]
)
sludge_emb_df = pd.DataFrame(
    df['sludge_dinov3_emb'].tolist(),
    columns = [f"sludge_emb_{i}" \
               for i in range(df["sludge_dinov3_emb"].iloc[0] \
                                                     .shape[0])
              ]
)

input_df = pd.concat([df[FEATURE_COLS].reset_index(drop = True), sludge_emb_df, lba_emb_df], axis = 1)
print("Input features are prepared.")
print(input_df.head())

output_df = df[TARGET_COLS].reset_index(drop = True)
print("Target variables are prepared.")
print(output_df.head())


Input features are prepared.
   interval_start  interval_end  sludge_emb_0  sludge_emb_1  sludge_emb_2  \
0            2675          2680     -0.444155     -0.257195      0.362337   
1            2680          2685     -0.134183     -0.076541     -0.063532   
2            2685          2690     -0.057407     -0.184069     -0.064041   
3            2690          2695     -0.082111     -0.067013     -0.096281   
4            2695          2700     -0.271062      0.104002     -0.005017   

   sludge_emb_3  sludge_emb_4  sludge_emb_5  sludge_emb_6  sludge_emb_7  ...  \
0      0.547643     -0.477527      0.593543      0.217066     -0.292527  ...   
1      0.363979     -0.575249      0.321932      0.319753      0.165333  ...   
2      0.224925     -0.339440      0.409055      0.112859      0.032745  ...   
3      0.436976     -0.512890      0.217838      0.152675      0.536699  ...   
4      0.572512     -0.429888      0.196399      0.251413      0.272011  ...   

   lba_emb_1014  lba_emb_10

## 5. GroupKFold

In [ ]:
groups = df["well_id"]
gkf = GroupKFold(n_splits = 3)

print("Amount of unique wells: ", df["well_id"].nunique())

Amount of unique wells:  4


## 6. Training

### 6.1. Model Preparation

In [8]:
params = {
    'iterations': 2500,
    'learning_rate': 0.05,
    'depth': 8,
    'loss_function': 'MultiRMSE',
    'eval_metric': 'MultiRMSE',
    'random_seed': 42,
    'early_stopping_rounds': 150,
    'verbose': 100,
    'task_type': 'CPU',
    'devices': '0'
}

### 6.2. Cross-Validation

In [ ]:
model = CatBoostRegressor(**params)

cv_scores = []
for fold, (train_idx, val_idx) in enumerate(gkf.split(input_df, output_df, groups)):
    input_train_data, input_val_data = input_df.iloc[train_idx], input_df.iloc[val_idx]
    output_train_data, output_val_data = output_df.iloc[train_idx], output_df.iloc[val_idx]

    model.fit(
        input_train_data, output_train_data,
        eval_set = (input_val_data, output_val_data),
        use_best_model = True,
        verbose = False
    )
    
    val_pred = model.predict(input_val_data)
    val_sum = val_pred.sum(axis = 1)
    score = np.abs(val_sum - 100).mean()

    cv_scores.append(score)
    print(f"Fold {fold + 1}: Mean ABS deviation from 100% = {score:.3f}.")

print(f"\nCV Mean deviation: {np.mean(cv_scores):.3f} +/- {np.std(cv_scores):.3f}.")

Fold 1: Mean ABS deviation from 100% = 0.000.
Fold 2: Mean ABS deviation from 100% = 1.974.
Fold 3: Mean ABS deviation from 100% = 0.662.
Fold 4: Mean ABS deviation from 100% = 0.229.

CV Mean deviation: 0.716 +/- 0.764.


### 6.3. Final Training

In [10]:
def normalize_predictions(predictions):
    predictions_sum = predictions.sum(axis = 1, keepdims = True)
    return predictions / predictions_sum * 100

In [ ]:
final_model = CatBoostRegressor(**params)
final_model.fit(input_df, output_df, verbose = True)

model_name = MODEL_NAME.format(datetime.datetime.now().strftime("%Y-%m-%d_%H:%M:%S"))
model_output_path = os.path.join(BASE_FOLDER, OUTPUT_FOLDER, model_name)
final_model.save_model(model_output_path)

predictions = final_model.predict(input_df)
predictions_normalized = normalize_predictions(predictions)

print("Mean derivation from 100% after normalization: ", \
      np.abs(predictions_normalized.sum(axis = 1) - 100).mean()
     )

0:	learn: 47.8122928	total: 199ms	remaining: 8m 17s
1:	learn: 45.9526001	total: 407ms	remaining: 8m 28s
2:	learn: 44.2817771	total: 619ms	remaining: 8m 34s
3:	learn: 42.6258741	total: 834ms	remaining: 8m 40s
4:	learn: 41.0518631	total: 1.05s	remaining: 8m 45s
5:	learn: 39.6680507	total: 1.27s	remaining: 8m 48s
6:	learn: 38.2188133	total: 1.5s	remaining: 8m 55s
7:	learn: 36.8401442	total: 1.72s	remaining: 8m 55s
8:	learn: 35.5466560	total: 1.94s	remaining: 8m 57s
9:	learn: 34.2643319	total: 2.16s	remaining: 8m 57s
10:	learn: 32.9786089	total: 2.37s	remaining: 8m 56s
11:	learn: 31.8067889	total: 2.58s	remaining: 8m 55s
12:	learn: 30.7760937	total: 2.8s	remaining: 8m 55s
13:	learn: 29.7193282	total: 3.02s	remaining: 8m 55s
14:	learn: 28.7468598	total: 3.23s	remaining: 8m 54s
15:	learn: 27.7873980	total: 3.44s	remaining: 8m 54s
16:	learn: 26.9312356	total: 3.66s	remaining: 8m 54s
17:	learn: 26.0474359	total: 3.87s	remaining: 8m 54s
18:	learn: 25.2217173	total: 4.09s	remaining: 8m 53s
19:	l

## 7. Results

### 7.1. Base Results

In [17]:
final_df = pd.DataFrame(
    index = output_df.index
)

predictions_normalized_df = pd.DataFrame(
    predictions_normalized,
    columns = output_df.columns,
    index = output_df.index
)

for column in output_df.columns:
    print(column)
    final_df[f"{column}-original"] = df[column]
    final_df[f"{column}-predicted"] = predictions_normalized_df[column]
    final_df[f"{column}-error"] = np.abs(df[column] - predictions_normalized_df[column])

print(final_df.head())
final_df.to_csv(os.path.join(BASE_FOLDER, OUTPUT_FOLDER, "results-base.csv"))

sandstone_sludge
siltstone_sludge
argillite_sludge
   sandstone_sludge-original  sandstone_sludge-predicted  \
0                          5                    5.013527   
1                          5                    5.014484   
2                          5                    4.972277   
3                          5                    5.051488   
4                          5                    4.992631   

   sandstone_sludge-error  siltstone_sludge-original  \
0                0.013527                         25   
1                0.014484                         25   
2                0.027723                         30   
3                0.051488                         30   
4                0.007369                         25   

   siltstone_sludge-predicted  siltstone_sludge-error  \
0                   24.992465                0.007535   
1                   25.016363                0.016363   
2                   30.019445                0.019445   
3                   30.

### 7.2. Metrics

In [20]:
metrics = []
for column in predictions_normalized_df:
    metrics.append({
        "target": column,
        "MAE": mean_absolute_error(
            df[column],
            predictions_normalized_df[column]
        ),
        "RMSE": root_mean_squared_error(
            df[column],
            predictions_normalized_df[column]
        ),
        "R2": r2_score(
            df[column],
            predictions_normalized_df[column]
        )
    })
metrics_df = pd.DataFrame(metrics)

print(metrics_df)

             target       MAE      RMSE        R2
0  sandstone_sludge  0.094837  0.229064  0.999924
1  siltstone_sludge  0.083131  0.191151  0.999927
2  argillite_sludge  0.833651  2.130662  0.996402
